# 10 Position-wise Feed-Forward Networks

**Position-wise Feed-Forward Network:** $\;$ MLP $\,\mathbb{R}^{d_\text{model}}\to\mathbb{R}^{d_\text{model}}\,$ con una capa oculta de $\,d_{\text{ff}}=4\,d_{\text{model}}\,$ ReLUs, que se aplica por separado a cada vector de la secuencia de entrada
<!-- $$\operatorname{PWFFN}(X)=\begin{pmatrix}%
\operatorname{FFN}(X_{1,:})\\\vdots\\\operatorname{FFN}(X_{n,:})
\end{pmatrix}\quad\text{con}\quad%
\operatorname{FFN}(\boldsymbol{x})%
=\operatorname{ReLU}(\boldsymbol{x}W_1+\boldsymbol{b}_1)W_2+\boldsymbol{b}_2$$ -->
$$\operatorname{PWFFN}(X)=\operatorname{ReLU}(XW_1+\boldsymbol{1}_n\boldsymbol{b}_1^t)W_2+\boldsymbol{1}_n\boldsymbol{b}_2^t$$


**Ejercicio:** $\;$ halla $\,\operatorname{PWFFN}(X)_{1,:}\,$ con $\,X=\begin{pmatrix}-0.7071&0.7071\\0.7071&-0.7071\\0.7070&-0.7070\end{pmatrix}\,$ y
$$\begin{align*}
W_1&=\begin{pmatrix}
0.4008 & -0.4451 & 0.7679 & -0.7363 & 0.2594 & 0.4195 & 0.2920 & -0.0160 \\
0.1917 & -0.6482 & 0.5881 & -0.6416 & 0.4606 & -0.2898 & 0.0965 & 0.0162
\end{pmatrix}\\
\boldsymbol{b}_1^t&=(0.0312,  0.2093,  0.2466, -0.5398, -0.3994,  0.3540,  0.4932, -0.2173)\\
W_2&=\begin{pmatrix}
0.5994&-0.2837&-0.2077&-0.5024&-0.5487&0.7268&0.6768&-0.6624\\
-0.4707&0.2907&0.2848&0.4173&0.4015&0.4828&0.1108&0.1021
\end{pmatrix}^t\\
\boldsymbol{b}_2^t&=(0.0928, -0.2395)
\end{align*}$$


<p style="page-break-after:always;"></p>


**Solución:**

In [1]:
import torch; import torch.nn as nn; torch.manual_seed(23)
import import_ipynb; from at251 import create_model
model = create_model(src_vocab_size=3, tgt_vocab_size=3, embed_dim=2, num_layers=1, num_heads=1, dropout=0.)
src = torch.LongTensor([[1, 2, 1]]); x = model.src_embed(src).data
norm_x = model.encoder.layers[0].norm_self_attn(x).data
x = x + model.encoder.layers[0].self_attn(norm_x, norm_x, norm_x).data
norm_x = model.encoder.layers[0].norm_ff(x); norm_x.data

tensor([[[-0.7071,  0.7071],
         [ 0.7071, -0.7071],
         [-0.7070,  0.7070]]])

In [2]:
ff = model.encoder.layers[0].ff
print("linear1 weight", ff.linear1.weight.data)
print("linear1 bias", ff.linear1.bias.data)
print("linear2 weight", ff.linear2.weight.data)
print("linear2 bias", ff.linear2.bias.data)

linear1 weight tensor([[ 0.4008,  0.1917],
        [-0.4451, -0.6482],
        [ 0.7679,  0.5881],
        [-0.7363, -0.6416],
        [ 0.2594,  0.4606],
        [ 0.4195, -0.2898],
        [ 0.2920,  0.0965],
        [-0.0160,  0.0162]])
linear1 bias tensor([ 0.0312,  0.2093,  0.2466, -0.5398, -0.3994,  0.3540,  0.4932, -0.2173])
linear2 weight tensor([[ 0.5994, -0.2837, -0.2077, -0.5024, -0.5487,  0.7268,  0.6768, -0.6624],
        [-0.4707,  0.2907,  0.2848,  0.4173,  0.4015,  0.4828,  0.1108,  0.1021]])
linear2 bias tensor([ 0.0928, -0.2395])



<p style="page-break-after:always;"></p>


In [3]:
norm_x0 = norm_x[0, 0].data; print(norm_x0)
linear1_norm_x0 = ff.linear1(norm_x0).data; print(linear1_norm_x0)
relu_norm_x0 = nn.functional.relu(linear1_norm_x0); print(relu_norm_x0)
linear2_norm_x0 = ff.linear2(relu_norm_x0).data; print(linear2_norm_x0)

tensor([-0.7071,  0.7071])
tensor([-0.1167,  0.0657,  0.1195, -0.4728, -0.2571, -0.1476,  0.3549, -0.1946])
tensor([0.0000, 0.0657, 0.1195, 0.0000, 0.0000, 0.0000, 0.3549, 0.0000])
tensor([ 0.2896, -0.1471])


$$\begin{align*}
\operatorname{PWFFN}(X)_{1,:}%
&=\operatorname{ReLU}((-0.7071, 0.7071)W_1+\boldsymbol{b}_1^t)W_2+\boldsymbol{b}_2^t\\
&=\operatorname{ReLU}(-0.1167,  0.0657,  0.1195, -0.4728, -0.2571, -0.1476,  0.3549, -0.1946)W_2+\boldsymbol{b}_2^t\\
&=(0.0000, 0.0657, 0.1195, 0.0000, 0.0000, 0.0000, 0.3549, 0.0000)W_2+\boldsymbol{b}_2^t\\
&=(0.2896, -0.1470)
\end{align*}$$

In [4]:
linear1_norm_x = ff.linear1(norm_x).data; # print(linear1_norm_x)
relu_norm_x = nn.functional.relu(linear1_norm_x); # print(relu_norm_x)
linear2_norm_x = ff.linear2(relu_norm_x).data; print(linear2_norm_x)

tensor([[[ 0.2896, -0.1471],
         [ 1.0716,  0.3682],
         [ 0.2896, -0.1470]]])


In [5]:
ff(norm_x).data

tensor([[[ 0.2896, -0.1471],
         [ 1.0716,  0.3682],
         [ 0.2896, -0.1470]]])


<p style="page-break-after:always;"></p>
